# Strength of Schedule Analysis

**Question**: does a player's schedule difficulty (how good/bad their
opponents' defenses are *against that player's position*) help explain
outcomes beyond what ADP captures?

**Approach**: build a position-specific "fantasy points allowed per game"
metric for every team-season using actual weekly stats (not generic
win/loss records -- a defense can be great vs. the run and bad vs. the
pass, which a single team rating would hide). Then, for each player,
average the difficulty of the opponents their team actually faced, and
check whether that adds signal beyond ADP.


In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

pd.set_option('display.max_columns', 100)
sns.set_style('whitegrid')

schedules = pd.read_parquet('../data/raw/schedules.parquet')
weekly = pd.read_parquet('../data/raw/weekly_player_stats.parquet')
adp_with_outcomes = pd.read_parquet('../data/processed/adp_with_outcomes.parquet')

print(f"schedules: {schedules.shape}")
print(f"weekly: {weekly.shape}")


schedules: (2743, 46)
weekly: (54479, 53)


## Step 1: convert schedules to team-week-opponent long format

In [15]:
reg_season = schedules[schedules['game_type'] == 'REG'].copy()

home_rows = reg_season[['season', 'week', 'home_team', 'away_team']].rename(
    columns={'home_team': 'team', 'away_team': 'opponent'}
)
away_rows = reg_season[['season', 'week', 'away_team', 'home_team']].rename(
    columns={'away_team': 'team', 'home_team': 'opponent'}
)

team_week_opponent = pd.concat([home_rows, away_rows], ignore_index=True)
print(f"team_week_opponent: {team_week_opponent.shape}")
team_week_opponent.head(10)


team_week_opponent: (5246, 4)


,season,week,team,opponent
0,2015,1,NE,PIT
1,2015,1,BUF,IND
2,2015,1,CHI,GB
3,2015,1,HOU,KC
4,2015,1,JAX,CAR
5,2015,1,NYJ,CLE
6,2015,1,STL,SEA
7,2015,1,WAS,MIA
8,2015,1,ARI,NO
9,2015,1,SD,DET


## Step 2: position-specific points allowed per team-week

In [16]:
SKILL_POSITIONS = ['QB', 'RB', 'WR', 'TE']
weekly_skill = weekly[weekly['position'].isin(SKILL_POSITIONS)]

points_allowed_weekly = weekly_skill.groupby(
    ['opponent_team', 'season', 'week', 'position']
)['fantasy_points_ppr'].sum().reset_index()
points_allowed_weekly = points_allowed_weekly.rename(columns={'opponent_team': 'team'})

print(f"points_allowed_weekly: {points_allowed_weekly.shape}")
points_allowed_weekly.head(10)


points_allowed_weekly: (21900, 5)


,team,season,week,position,fantasy_points_ppr
0,ARI,2015,1,QB,16.500000
1,ARI,2015,1,RB,32.200001
2,ARI,2015,1,TE,4.900000
3,ARI,2015,1,WR,36.600002
4,ARI,2015,2,QB,11.540000
5,ARI,2015,2,RB,23.400002
6,ARI,2015,2,TE,12.200001
7,ARI,2015,2,WR,31.500000
8,ARI,2015,3,QB,5.280000
9,ARI,2015,3,RB,9.200000


## Step 3: aggregate to team-season points allowed per game, by position

Lower value = tougher defense against that position that season.


In [17]:
points_allowed_season = points_allowed_weekly.groupby(
    ['team', 'season', 'position']
)['fantasy_points_ppr'].mean().reset_index()
points_allowed_season = points_allowed_season.rename(
    columns={'fantasy_points_ppr': 'points_allowed_per_game'}
)

print(f"points_allowed_season: {points_allowed_season.shape}")
points_allowed_season.sort_values('points_allowed_per_game').head(10)


points_allowed_season: (1280, 4)


,team,season,position,points_allowed_per_game
6,ARI,2016,TE,7.173333
1006,PHI,2016,TE,7.293334
910,NO,2022,TE,7.376471
866,NE,2021,TE,7.661111
26,ARI,2021,TE,7.761111
134,BUF,2018,TE,7.762500
98,BAL,2019,TE,8.170588
974,NYJ,2018,TE,8.262500
1142,SF,2020,TE,8.343750
386,DEN,2021,TE,8.617647


## Step 4: player-team-season SOS metric

Join each team's weekly opponents to what that opponent allows to each
position over the full season, then average across weeks -- giving each
team-season-position combo a single "average difficulty of opponents
faced" number.


In [18]:
schedule_difficulty = team_week_opponent.merge(
    points_allowed_season,
    left_on=['opponent', 'season'],
    right_on=['team', 'season'],
    suffixes=('', '_opp'),
)
schedule_difficulty = schedule_difficulty.drop(columns=['team_opp'])

team_season_position_sos = schedule_difficulty.groupby(
    ['team', 'season', 'position']
)['points_allowed_per_game'].mean().reset_index()
team_season_position_sos = team_season_position_sos.rename(
    columns={'points_allowed_per_game': 'sos_points_allowed'}
)

print(f"team_season_position_sos: {team_season_position_sos.shape}")
team_season_position_sos.sort_values('sos_points_allowed').head(10)


team_season_position_sos: (1280, 4)


,team,season,position,sos_points_allowed
650,LA,2018,TE,10.648171
782,MIN,2018,TE,10.662571
454,GB,2018,TE,10.807674
414,DET,2018,TE,10.946694
214,CHI,2018,TE,11.039315
70,ATL,2022,TE,11.049034
1190,TB,2022,TE,11.050162
1102,SEA,2021,TE,11.101357
918,NYG,2022,TE,11.208947
758,MIA,2022,TE,11.283387


## Step 5: attach each player's team, then join in their SOS

`adp_with_outcomes` doesn't reliably have team (many ADP rows had team
blank). Instead, use each player's most common `recent_team` from `weekly`
that season (handles most trade cases reasonably).


In [19]:
player_team_season = (
    weekly.groupby(['player_id', 'season'])['recent_team']
    .agg(lambda x: x.mode().iloc[0])
    .reset_index()
    .rename(columns={'recent_team': 'team'})
)

# adp_with_outcomes already has its own (mostly-blank) 'team' column from the
# original ADP parsing -- drop it here since we're replacing it with the
# more reliable weekly-derived team below.
sos_data = adp_with_outcomes.drop(columns=['team']).merge(
    player_team_season, on=['player_id', 'season'], how='left'
)
sos_data = sos_data.merge(
    team_season_position_sos,
    on=['team', 'season', 'position'],
    how='left',
)

print(f"sos_data: {sos_data.shape}")
print(f"Rows missing SOS: {sos_data['sos_points_allowed'].isna().sum()}")
sos_data[['player_name', 'season', 'position', 'team', 'adp_avg', 'fantasy_points_ppr', 'sos_points_allowed']].head(10)


sos_data: (4673, 20)
Rows missing SOS: 846


,player_name,season,position,team,adp_avg,fantasy_points_ppr,sos_points_allowed
0,Le'Veon Bell,2015,RB,PIT,1.5,111.2,20.053633
1,Adrian Peterson,2015,RB,MIN,1.8,260.7,21.852802
2,Antonio Brown,2015,WR,PIT,3.8,390.2,35.506660
3,Jamaal Charles,2015,RB,KC,4.0,101.1,20.050249
4,Eddie Lacy,2015,RB,GB,4.0,140.6,21.271955
5,Marshawn Lynch,2015,RB,SEA,7.3,82.7,21.589039
6,Julio Jones,2015,WR,ATL,7.8,369.1,35.537346
7,Dez Bryant,2015,WR,DAL,8.0,89.1,35.357185
8,Rob Gronkowski,2015,TE,NE,9.0,255.6,13.483751
9,Odell Beckham Jr.,2015,WR,NYG,11.0,319.3,35.132580


## Step 6: does SOS add signal beyond ADP?

*(Note: 846 rows are missing SOS, ~18% -- likely team-abbreviation
mismatches for relocated franchises like OAK/LV, SD/LAC, STL/LA between
`schedules` and `weekly`. Not chasing this down further for now since this
is a stretch analysis; flagging as a known gap.)*

**Test**: fit a simple ADP -> outcome relationship per position, take the
residual (how much a player over/under-performed relative to what ADP alone
predicts), then check whether that residual correlates with SOS. If SOS
explains real additional variance, we'd see a meaningful correlation between
the residual and `sos_points_allowed` (positive: easier schedule -> player
outperformed their ADP-implied expectation more than predicted).


In [20]:
from scipy.stats import linregress

sos_clean = sos_data.dropna(subset=['adp_avg', 'fantasy_points_ppr', 'sos_points_allowed']).copy()

results = []
for pos in ['QB', 'RB', 'WR', 'TE']:
    subset = sos_clean[sos_clean['position'] == pos].copy()

    # Fit fantasy_points_ppr ~ adp_avg, get residuals (actual - predicted)
    slope, intercept, r_value, p_value, std_err = linregress(subset['adp_avg'], subset['fantasy_points_ppr'])
    subset['predicted_by_adp'] = slope * subset['adp_avg'] + intercept
    subset['residual'] = subset['fantasy_points_ppr'] - subset['predicted_by_adp']

    # Does the residual correlate with SOS?
    corr, pval = spearmanr(subset['sos_points_allowed'], subset['residual'])
    results.append({'position': pos, 'n': len(subset), 'adp_r_squared': r_value**2,
                     'sos_vs_residual_corr': corr, 'p_value': pval})

    sos_clean.loc[subset.index, 'residual'] = subset['residual']

results_df = pd.DataFrame(results)
results_df


,position,n,adp_r_squared,sos_vs_residual_corr,p_value
0,QB,550,0.510107,0.064189,0.132714
1,RB,1136,0.374412,0.019488,0.511712
2,WR,1443,0.409686,0.033808,0.199318
3,TE,692,0.389622,0.096124,0.011409
